<a href="https://colab.research.google.com/github/andandandand/practical-computer-vision/blob/main/ASL_from_Kaggle_to_FiftyOne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASL-MNIST: From Kaggle to FiftyOne in a Single Notebook

#### Author: [Antonio Rueda-Toicen](antonio@getfiftyone.com)



[![Creative Commons License](https://i.creativecommons.org/l/by/4.0/88x31.png)](http://creativecommons.org/licenses/by/4.0/)

This work is licensed under a [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).


This notebook provides an end-to-end workflow for:
1. Downloading the American Sign Language (ASL) MNIST dataset from Kaggle.
2. Processing the raw CSV data into individual image files.
3. Creating a rich, explorable FiftyOne dataset from these images.
4. Visualizing and querying the dataset to understand its properties.

## 1. Setup and Installation

First, we'll install the necessary Python libraries. We need `fiftyone` for dataset management and visualization, and `kaggle` to download the data directly.

In [4]:
!pip install fiftyone==1.7.0 kaggle==1.7.4.5 > /dev/null

## 2. Kaggle API Configuration

To download datasets from Kaggle, you need to authenticate using your Kaggle API token.

1. Go to your Kaggle account page.
2. Click on 'Create New API Token'. This will download a `kaggle.json` file.
3. Run the cell below and upload that `kaggle.json` file.

In [5]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"andandand","key":"91ccc18b83337fe30927a6790b13efcc"}'}

In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 3. Data Download and Preparation

Now we'll download the ASL-MNIST dataset from Kaggle. The data comes in CSV files, so we'll need to:

1. Download and unzip the dataset.
2. Create directories to store the train and test images.
3. Read the CSV files using pandas.
4. Convert the pixel data in each row into a 28x28 JPG image and save it to the appropriate directory.

In [7]:
!kaggle datasets download -d datamunge/sign-language-mnist
!unzip sign-language-mnist.zip

Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
  0% 0.00/62.6M [00:00<?, ?B/s]
100% 62.6M/62.6M [00:00<00:00, 2.09GB/s]
Archive:  sign-language-mnist.zip
  inflating: amer_sign2.png          
  inflating: amer_sign3.png          
  inflating: american_sign_language.PNG  
  inflating: sign_mnist_test.csv     
  inflating: sign_mnist_test/sign_mnist_test.csv  
  inflating: sign_mnist_train.csv    
  inflating: sign_mnist_train/sign_mnist_train.csv  


In [8]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

data_path = Path('asl_mnist_data')
train_images_dir = data_path / 'train'
test_images_dir = data_path / 'test'

# Create directories for the images
train_images_dir.mkdir(parents=True, exist_ok=True)
test_images_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv('sign_mnist_train/sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test/sign_mnist_test.csv')

def process_and_save_images(df, image_dir, split_name):
    print(f'Processing and saving {split_name} images...')
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        label = row['label']
        pixels = row[1:].values.reshape(28, 28).astype(np.uint8)
        image = Image.fromarray(pixels)
        image.save(image_dir / f'{split_name}_{index}.jpg')

process_and_save_images(train_df, train_images_dir, 'train')
process_and_save_images(test_df, test_images_dir, 'test')

Processing and saving train images...


  0%|          | 0/27455 [00:00<?, ?it/s]

Processing and saving test images...


  0%|          | 0/7172 [00:00<?, ?it/s]

## 4. FiftyOne Dataset Creation

With our images saved as JPG files, we can now create a FiftyOne dataset. This will allow us to easily visualize, query, and analyze our data.

In [9]:
import fiftyone as fo

# Create or load the FiftyOne dataset
if fo.dataset_exists("asl-mnist"):
    asl_dataset = fo.load_dataset("asl-mnist")
    asl_dataset.delete()
asl_dataset = fo.Dataset("asl-mnist")
asl_dataset.persistent = True

# ASL alphabet mapping (excluding J=9 and Z=25 which require motion)
asl_labels = {
    0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I',
    10: 'K', 11: 'L', 12: 'M', 13: 'N', 14: 'O', 15: 'P', 16: 'Q', 17: 'R',
    18: 'S', 19: 'T', 20: 'U', 21: 'V', 22: 'W', 23: 'X', 24: 'Y'
}

def create_fiftyone_samples(df, image_dir, split_name):
    samples = []
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        label_idx = row['label']
        image_path = image_dir / f'{split_name}_{index}.jpg'

        label_letter = asl_labels.get(label_idx, f'unknown_{label_idx}')

        sample = fo.Sample(
            filepath=str(image_path),
            tags=[split_name],
            ground_truth=fo.Classification(label=label_letter)
        )
        samples.append(sample)
    return samples

train_samples = create_fiftyone_samples(train_df, train_images_dir, 'train')
test_samples = create_fiftyone_samples(test_df, test_images_dir, 'test')

asl_dataset.add_samples(train_samples)
asl_dataset.add_samples(test_samples)

print(f"Dataset created with {len(asl_dataset)} samples.")

  0%|          | 0/27455 [00:00<?, ?it/s]

  0%|          | 0/7172 [00:00<?, ?it/s]

 100% |█████████████| 27455/27455 [9.0s elapsed, 0s remaining, 2.9K samples/s]      


INFO:eta.core.utils: 100% |█████████████| 27455/27455 [9.0s elapsed, 0s remaining, 2.9K samples/s]      


 100% |███████████████| 7172/7172 [2.1s elapsed, 0s remaining, 3.3K samples/s]      


INFO:eta.core.utils: 100% |███████████████| 7172/7172 [2.1s elapsed, 0s remaining, 3.3K samples/s]      


Dataset created with 34627 samples.


## Compute metadata

In [10]:
asl_dataset.compute_metadata()

Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |█████████████| 34627/34627 [1.0m elapsed, 0s remaining, 772.2 samples/s]      


INFO:eta.core.utils: 100% |█████████████| 34627/34627 [1.0m elapsed, 0s remaining, 772.2 samples/s]      


## 5. Data Exploration and Visualization

Now that our dataset is in FiftyOne, we can launch the FiftyOne App to explore it. We'll also demonstrate how to filter for the 'unknown' labels which correspond to the motion-based signs 'J' and 'Z'.

In [19]:
session = fo.launch_app(asl_dataset, auto=False)
print(f"Session URL: {session.url}")

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.


Session URL: https://5151-m-hm-r8prn15hswo6-c.europe-west4-0.prod.colab.dev?polling=true


In [17]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [18]:
from fiftyone.utils.huggingface import push_to_hub

push_to_hub(asl_dataset,
            "American-Sign-Language-MNIST",
            repo_type="dataset",
            license="mit",
            exist_ok=True,
            chunk_size=100)

Directory '/tmp/tmpiguz4gxp' already exists; export will be merged with existing files


Exporting samples...


INFO:fiftyone.utils.data.exporters:Exporting samples...


 100% |████████████████| 34627/34627 [6.7s elapsed, 0s remaining, 5.3K docs/s]       


INFO:eta.core.utils: 100% |████████████████| 34627/34627 [6.7s elapsed, 0s remaining, 5.3K docs/s]       
Uploading media files in 347 batches of size 100:   0%|          | 0/347 [00:00<?, ?it/s]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 347 batches of size 100:   0%|          | 1/347 [00:01<10:45,  1.86s/it]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 347 batches of size 100:   1%|          | 2/347 [00:02<07:35,  1.32s/it]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 347 batches of size 100:   1%|          | 3/347 [00:04<07:19,  1.28s/it]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading media files in 347 batches of size 100:   1%|          | 4/347 [00:05<06:54,  1.21s/it]No files have been modified since last commit. Skipping to prevent empty commit.
Uploading me